# Load Data (Eurostat API)

## Inflation

In [2]:
import requests

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_manr?lang=EN"
params = {"lang":"EN"}

r = requests.get(url, params = params, timeout = 30)
r.raise_for_status()

data = r.json()
print(type(data))

<class 'dict'>


In [3]:
print(data.keys())

dict_keys(['version', 'class', 'label', 'source', 'updated', 'value', 'status', 'id', 'size', 'dimension', 'extension'])


### Exploring data

In [6]:
print(data.get("id"))

['freq', 'unit', 'coicop', 'geo', 'time']


In [7]:
print(data.get("label"))

HICP - monthly data (annual rate of change)


In [8]:
print(data.get("version"))

2.0


In [9]:
print(data.get("class"))

dataset


In [10]:
print(data.get("updated"))

2026-02-06T23:00:00+0100


In [11]:
print(data.get("size"))

[1, 1, 467, 45, 348]


---

### Filter countries:

#### ES, DE, FR, IT

### Filter time range:

#### 2015-01 a 2023-12

In [17]:

params = [
    ("lang", "EN"),
    ("geo", "ES"),
    ("geo", "DE"),
    ("geo", "FR"),
    ("geo", "IT"),   
    ("sinceTimePeriod", "2015-01"),  
    ("untilTimePeriod", "2023-12"),  
]

r = requests.get(url, params =params)
data = r.json()


In [18]:
print(data.keys())

dict_keys(['version', 'class', 'label', 'source', 'updated', 'value', 'status', 'id', 'size', 'dimension', 'extension'])


In [19]:
print(data.get("size"))  #Here we can see that the data got reduced accordingly to our scope

[1, 1, 467, 4, 108]


---

### Make the data a table (JSON-stat)

#### Double checked the keys for creating the table

In [20]:
print(data["dimension"].keys())

dict_keys(['freq', 'unit', 'coicop', 'geo', 'time'])


In [21]:
print(data["dimension"]["geo"].keys())

dict_keys(['label', 'category'])


In [22]:
print(data["dimension"]["time"].keys())

dict_keys(['label', 'category'])


#### Get the categories to ensure all the info is still there

In [24]:
ids = data["id"]

for dimension in ids:
    print("\nDIMENSION:", dimension)
    print("codes:", list(data["dimension"][dimension]["category"]["index"].keys())[:10])


DIMENSION: freq
codes: ['M']

DIMENSION: unit
codes: ['RCH_A']

DIMENSION: coicop
codes: ['CP00', 'CP01', 'CP011', 'CP0111', 'CP01111', 'CP01112', 'CP01113', 'CP01114', 'CP01115', 'CP01116']

DIMENSION: geo
codes: ['DE', 'ES', 'FR', 'IT']

DIMENSION: time
codes: ['2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10']


In [ ]:
'''
Call the API again and filter the CP00 to save the final data frame
'''

In [25]:
params = [
    ("lang", "EN"),
    ("geo", "ES"),
    ("geo", "DE"),
    ("geo", "FR"),
    ("geo", "IT"),
    ("coicop", "CP00"),
    ("sinceTimePeriod", "2015-01"),
    ("untilTimePeriod", "2023-12"),
]

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data_inflation = response.json()

In [26]:
print(data_inflation.keys())

dict_keys(['version', 'class', 'label', 'source', 'updated', 'value', 'id', 'size', 'dimension', 'extension'])


In [27]:
print(data_inflation.get("size"))

[1, 1, 1, 4, 108]


In [28]:
print(data_inflation.get("id"))

['freq', 'unit', 'coicop', 'geo', 'time']


In [30]:
ids = data_inflation["id"]

for dimension in ids:
    print("\nDIMENSION:", dimension)
    print("codes:", list(data_inflation["dimension"][dimension]["category"]["index"].keys())[:10])


DIMENSION: freq
codes: ['M']

DIMENSION: unit
codes: ['RCH_A']

DIMENSION: coicop
codes: ['CP00']

DIMENSION: geo
codes: ['DE', 'ES', 'FR', 'IT']

DIMENSION: time
codes: ['2015-01', '2015-02', '2015-03', '2015-04', '2015-05', '2015-06', '2015-07', '2015-08', '2015-09', '2015-10']


#### Create Dataframe

In [32]:
import pandas as pd
import itertools

ids = data_inflation["id"]
values_obj = data_inflation["value"]

#Obtain codes sorted according to index

dim_codes = {}

for dim in ids:
    cat_index = data["dimension"][dim]["category"]["index"]
    ordered_codes = [code for code, value in sorted(cat_index.items(),key=lambda x:x[1])]
    dim_codes[dim] = ordered_codes

#Generate the table with all the possible combinations
all_combinations = list(itertools.product(*(dim_codes[dim] for dim in ids)))

#Build each row

rows = []

for i, combo in enumerate (all_combinations):
    row = dict(zip(ids,combo))

    if isinstance(values_obj, dict):
        row["value"]= values_obj.get(str(i))
    else:
        row["value"]= values_obj[i] if i < len(values_obj) else None

    rows.append(row)

df = pd.DataFrame(rows)

In [33]:
print(df.head())

  freq   unit coicop geo     time  value
0    M  RCH_A   CP00  DE  2015-01   -0.5
1    M  RCH_A   CP00  DE  2015-02   -0.2
2    M  RCH_A   CP00  DE  2015-03    0.3
3    M  RCH_A   CP00  DE  2015-04    1.0
4    M  RCH_A   CP00  DE  2015-05    1.6


In [34]:
print(df.shape)

(201744, 6)


---

## Clean dataset in pandas

In [37]:
#Rename columns

df = df.rename(columns = {
    "geo": "country",
    "time": "date",
    "value": "inflation"
})

In [38]:
df.head()

,freq,unit,coicop,country,date,inflation
0,M,RCH_A,CP00,DE,2015-01,-0.5
1,M,RCH_A,CP00,DE,2015-02,-0.2
2,M,RCH_A,CP00,DE,2015-03,0.3
3,M,RCH_A,CP00,DE,2015-04,1.0
4,M,RCH_A,CP00,DE,2015-05,1.6


In [40]:
#Save only Date, Country and Inflation

df = df[["date","country","inflation"]].copy()

df.head(10)

,date,country,inflation
0,2015-01,DE,-0.5
1,2015-02,DE,-0.2
2,2015-03,DE,0.3
3,2015-04,DE,1.0
4,2015-05,DE,1.6
5,2015-06,DE,1.1
6,2015-07,DE,1.3
7,2015-08,DE,1.1
8,2015-09,DE,0.8
9,2015-10,DE,1.2


In [41]:
df["country"].unique()

array(['DE', 'ES', 'FR', 'IT'], dtype=object)

In [43]:
#Standardize names inside country, dates and inflation rates
df["date"] = pd.to_datetime(df["date"], errors = "coerce").dt.to_period("M").astype(str)
df["country"] = df["country"].str.upper()
df["inflation"] = pd.to_numeric(df["inflation"], errors = "coerce")

In [47]:
#Clean nules
df.dropna(subset = ["date","country","inflation"])
#Clean duplicates
df.drop_duplicates(subset = ["date","country"])
#Sort values per country and date
df.sort_values(["country","date"]).reset_index(drop=True)


,date,country,inflation
0,2015-01,DE,-0.5
1,2015-01,DE,NaN
2,2015-01,DE,NaN
3,2015-01,DE,NaN
4,2015-01,DE,NaN
...,...,...,...
201739,2023-12,IT,NaN
201740,2023-12,IT,NaN
201741,2023-12,IT,NaN
201742,2023-12,IT,NaN


In [49]:
print(df.shape)

(201744, 3)


In [50]:
df.drop_duplicates(subset = ["date","country"])

,date,country,inflation
0,2015-01,DE,-0.5
1,2015-02,DE,-0.2
2,2015-03,DE,0.3
3,2015-04,DE,1.0
4,2015-05,DE,1.6
...,...,...,...
427,2023-08,IT,5.5
428,2023-09,IT,5.6
429,2023-10,IT,1.8
430,2023-11,IT,0.6


## Save Dataset clean

In [52]:
df.to_csv("inflation_clean.csv", index=False)